In [1]:
import sqlite3
import pandas as pd
import numpy as np
import os

In [ ]:
import pandas as pd
from pgmpy.models import DynamicBayesianNetwork as DBN
from pgmpy.estimators import MaximumLikelihoodEstimator
from pgmpy.inference import DBNInference

In [57]:
source_db = 'bnpl_proxy_data.db'
target_db = 'bnpl_proxy_data_age_change.db'
source_conn = sqlite3.connect(source_db)
df_raw = pd.read_sql_query("SELECT * FROM bnpl_loans_with_age_change", source_conn)
df_raw.to_csv('bnpl_loans_with_age_change.csv', index=False)
source_conn.close()


In [41]:
data =df_raw
static_cols = [
    'loan_amnt','grade','sub_grade','emp_length','home_ownership','annual_inc',
    'delinq_2yrs','inq_last_6mths','mths_since_last_delinq',
    'mths_since_last_record','open_acc','total_acc',
    'initial_list_status','policy_code','application_type','annual_inc_joint',
    'dti_joint','verification_status_joint','acc_now_delinq','tot_coll_amt','tot_cur_bal',
    'open_acc_6m','open_act_il','open_il_12m','open_il_24m','mths_since_rcnt_il',
    'total_bal_il','il_util','open_rv_12m','open_rv_24m','max_bal_bc','all_util',
    'total_rev_hi_lim','inq_fi','total_cu_tl','inq_last_12m','acc_open_past_24mths',
    'avg_cur_bal','bc_open_to_buy','bc_util','chargeoff_within_12_mths','delinq_amnt',
    'mo_sin_old_il_acct','mo_sin_old_rev_tl_op','mo_sin_rcnt_rev_tl_op','mo_sin_rcnt_tl',
    'mort_acc','mths_since_recent_bc','mths_since_recent_bc_dlq','mths_since_recent_inq',
    'mths_since_recent_revol_delinq','num_accts_ever_120_pd','num_actv_bc_tl',
    'num_actv_rev_tl','num_bc_sats','num_bc_tl','num_il_tl','num_op_rev_tl','num_rev_accts',
    'num_rev_tl_bal_gt_0','num_sats','num_tl_120dpd_2m','num_tl_30dpd','num_tl_90g_dpd_24m',
    'num_tl_op_past_12m','pct_tl_nvr_dlq','percent_bc_gt_75','pub_rec_bankruptcies','tax_liens',
    'tot_hi_cred_lim','total_bal_ex_mort','total_bc_limit','total_il_high_credit_limit',
    'issue_quarter','approx_age','age_group'
]
dynamic_cols = [
    'total_pymnt_t1','total_pymnt_t2','total_pymnt_t3','total_pymnt_t4','total_pymnt_t5','total_pymnt_t6',
    'revol_bal_t1','revol_bal_t2','revol_bal_t3','revol_bal_t4','revol_bal_t5','revol_bal_t6',
    'last_pymnt_amnt_t1','last_pymnt_amnt_t2','last_pymnt_amnt_t3','last_pymnt_amnt_t4','last_pymnt_amnt_t5','last_pymnt_amnt_t6',
    'default_indicator_t1','default_indicator_t2','default_indicator_t3','default_indicator_t4','default_indicator_t5','default_indicator_t6',
    'fico_score_t1','fico_score_t2','fico_score_t3','fico_score_t4','fico_score_t5','fico_score_t6',
    'revol_util_t1','revol_util_t2','revol_util_t3','revol_util_t4','revol_util_t5','revol_util_t6',
    'out_prncp_t1','out_prncp_t2','out_prncp_t3','out_prncp_t4','out_prncp_t5','out_prncp_t6',
    'delinq_count_t1','delinq_count_t2','delinq_count_t3','delinq_count_t4','delinq_count_t5','delinq_count_t6',
    'GDP_t1','gdp_growth_t1','unemployment_rate_t1','fedfunds_t1','inflation_t1','housing_price_t1',
    'GDP_t2','gdp_growth_t2','unemployment_rate_t2','fedfunds_t2','inflation_t2','housing_price_t2',
    'GDP_t3','gdp_growth_t3','unemployment_rate_t3','fedfunds_t3','inflation_t3','housing_price_t3',
    'GDP_t4','gdp_growth_t4','unemployment_rate_t4','fedfunds_t4','inflation_t4','housing_price_t4',
    'GDP_t5','gdp_growth_t5','unemployment_rate_t5','fedfunds_t5','inflation_t5','housing_price_t5',
    'GDP_t6','gdp_growth_t6','unemployment_rate_t6','fedfunds_t6','inflation_t6','housing_price_t6',
]

In [42]:
# -------------------------
# 3. Reshape the Data for DBN Input
# -------------------------
# (a) Create the static DataFrame for time slice 0.
# To avoid conflicts with dynamic features, we rename static columns by appending "_static".
static_df = data[static_cols].copy()
static_df.columns = [col + "_static" for col in static_df.columns]

# (b) Process dynamic columns.
# Remap so that _t1 becomes time slice 0, _t2 becomes 1, ..., _t6 becomes 5.
# We build a dictionary: key = new time slice, value = DataFrame for that time slice.
dynamic_dict = {}
for col in dynamic_cols:
    try:
        var, t = col.rsplit('_t', 1)
    except ValueError:
        continue
    if t in [str(i) for i in range(1, 7)]:
        new_t = int(t) - 1  # remap: t1 -> 0, t2 -> 1, etc.
        dynamic_dict.setdefault(new_t, {})[var] = data[col]

# Create a DataFrame for each dynamic time slice.
dynamic_dfs = {}
for t in sorted(dynamic_dict.keys()):
    df_t = pd.DataFrame(dynamic_dict[t])
    dynamic_dfs[t] = df_t

# (c) Build a slice-by-slice dictionary:
# For time slice 0, combine static features and dynamic features at time 0.
# For slices 1..5, use only dynamic features.
slices = {}
slices[0] = pd.concat([static_df, dynamic_dfs[0]], axis=1)
for t in range(1, 6):
    slices[t] = dynamic_dfs[t]

# (d) Convert each slice’s DataFrame so that its columns are a MultiIndex (variable, time).
def set_multiindex(df, t):
    df.columns = pd.MultiIndex.from_product([df.columns, [t]])
    return df

for t in slices:
    slices[t] = set_multiindex(slices[t], t)

# (e) Concatenate all slices along the column axis.
combined_df = pd.concat([slices[t] for t in sorted(slices.keys())], axis=1)

# Verify that the combined DataFrame’s columns start at time 0.
print("Combined DataFrame columns:")
print(combined_df.columns)

Combined DataFrame columns:
MultiIndex([(             'loan_amnt_static', 0),
            (                 'grade_static', 0),
            (             'sub_grade_static', 0),
            (            'emp_length_static', 0),
            (        'home_ownership_static', 0),
            (            'annual_inc_static', 0),
            (           'delinq_2yrs_static', 0),
            (        'inq_last_6mths_static', 0),
            ('mths_since_last_delinq_static', 0),
            ('mths_since_last_record_static', 0),
            ...
            (                   'fico_score', 5),
            (                   'revol_util', 5),
            (                    'out_prncp', 5),
            (                 'delinq_count', 5),
            (                          'GDP', 5),
            (                   'gdp_growth', 5),
            (            'unemployment_rate', 5),
            (                     'fedfunds', 5),
            (                    'inflation', 5),
      

In [43]:
model = DBN()

# (a) Add static nodes (from time slice 0, these have names ending with '_static').
for col in static_df.columns:
    model.add_node((col, 0))

# (b) Add dynamic nodes for each time slice.
# For dynamic nodes, we use the variable names from the dynamic part.
# (Assume that the set of dynamic variables is the same for all time slices.)
dynamic_vars = dynamic_dfs[0].columns.tolist()
for t in range(0, 6):
    for var in dynamic_vars:
        model.add_node((var, t))

# (c) Define intra-slice dependencies for dynamic nodes.
# (This is an example—adjust the structure as needed.)
for t in range(0, 6):
    if 'default_indicator' in dynamic_vars and 'fico_score' in dynamic_vars:
        model.add_edge(('fico_score', t), ('default_indicator', t))
    if 'default_indicator' in dynamic_vars and 'last_pymnt_amnt' in dynamic_vars:
        model.add_edge(('last_pymnt_amnt', t), ('default_indicator', t))
    if 'default_indicator' in dynamic_vars and 'revol_util' in dynamic_vars:
        model.add_edge(('revol_util', t), ('default_indicator', t))

# (d) Define inter-slice dependencies among dynamic nodes.
# For example, add an edge from each dynamic node at time t to itself at time t+1.
for t in range(0, 5):
    for var in dynamic_vars:
        model.add_edge((var, t), (var, t+1))
    if 'default_indicator' in dynamic_vars:
        model.add_edge(('default_indicator', t), ('default_indicator', t+1))

# (e) Link static nodes to the dynamic slice 0.
for s_var in static_df.columns:
    model.add_edge((s_var, 0), ('default_indicator', 0))

print("\nDBN structure (edges):")
print(model.edges())


DBN structure (edges):
[(<DynamicNode(fico_score, 0) at 0x362fdf880>, <DynamicNode(default_indicator, 0) at 0x311cc4400>), (<DynamicNode(fico_score, 0) at 0x362fdf880>, <DynamicNode(fico_score, 1) at 0x311cc4ca0>), (<DynamicNode(default_indicator, 0) at 0x311cc4400>, <DynamicNode(default_indicator, 1) at 0x311cc4be0>), (<DynamicNode(fico_score, 1) at 0x311cc4490>, <DynamicNode(default_indicator, 1) at 0x311cc44f0>), (<DynamicNode(last_pymnt_amnt, 0) at 0x311cc4550>, <DynamicNode(default_indicator, 0) at 0x311cc45b0>), (<DynamicNode(last_pymnt_amnt, 0) at 0x311cc4550>, <DynamicNode(last_pymnt_amnt, 1) at 0x311cc4af0>), (<DynamicNode(last_pymnt_amnt, 1) at 0x311cc4610>, <DynamicNode(default_indicator, 1) at 0x311cc4670>), (<DynamicNode(revol_util, 0) at 0x311cc46d0>, <DynamicNode(default_indicator, 0) at 0x311cc4730>), (<DynamicNode(revol_util, 0) at 0x311cc46d0>, <DynamicNode(revol_util, 1) at 0x311cc4d60>), (<DynamicNode(revol_util, 1) at 0x311cc4790>, <DynamicNode(default_indicator, 

In [32]:
combined_df.head()

,loan_amnt_static,grade_static,sub_grade_static,emp_length_static,home_ownership_static,annual_inc_static,delinq_2yrs_static,inq_last_6mths_static,mths_since_last_delinq_static,mths_since_last_record_static,...,fico_score,revol_util,out_prncp,delinq_count,GDP,gdp_growth,unemployment_rate,fedfunds,inflation,housing_price
,0,0,0,0,0,0,0,0,0,0,...,5,5,5,5,5,5,5,5,5,5
0,3600.0,C,C4,10+ years,MORTGAGE,55000.0,0.0,1.0,30.0,NaN,...,720.223477,0.000000,0.0,0.0,19089.379,3.548886,4.766667,0.45,2.050799,184.389
1,1400.0,C,C2,3 years,MORTGAGE,64000.0,0.0,0.0,NaN,NaN,...,792.798135,0.000000,0.0,0.0,19089.379,3.548886,4.766667,0.45,2.050799,184.389
2,5000.0,C,C3,2 years,RENT,79000.0,0.0,0.0,NaN,NaN,...,676.901820,0.000000,0.0,0.0,19089.379,3.548886,4.766667,0.45,2.050799,184.389
3,4225.0,C,C5,5 years,RENT,35000.0,2.0,0.0,18.0,NaN,...,648.800845,0.164031,0.0,4.0,19089.379,3.548886,4.766667,0.45,2.050799,184.389
4,2500.0,D,D3,10+ years,MORTGAGE,50000.0,3.0,4.0,6.0,NaN,...,712.206066,0.000000,0.0,0.0,19089.379,3.548886,4.766667,0.45,2.050799,184.389


In [44]:
# -------------------------
# 5. Fit the DBN Parameters
# -------------------------
# Fit the model on the combined DataFrame using Maximum Likelihood Estimation.
model.fit(combined_df)

# (Optional) Print out the learned CPDs
print("\nLearned CPDs:")
for cpd in model.get_cpds():
    print(cpd)



KeyError: '["(\'total_pymnt\', 1)_0", "(\'housing_price\', 3)_0", "(\'out_prncp\', 3)_0", "(\'mo_sin_old_il_acct_static\', 0)_0", "(\'pct_tl_nvr_dlq_static\', 0)_0", "(\'initial_list_status_static\', 0)_0", "(\'gdp_growth\', 4)_0", "(\'fico_score\', 4)_0", "(\'inflation\', 2)_0", "(\'mths_since_last_record_static\', 0)_0", "(\'num_op_rev_tl_static\', 0)_0", "(\'mths_since_recent_inq_static\', 0)_0", "(\'unemployment_rate\', 5)_0", "(\'chargeoff_within_12_mths_static\', 0)_0", "(\'GDP\', 0)_0", "(\'inq_fi_static\', 0)_0", "(\'total_pymnt\', 3)_0", "(\'housing_price\', 5)_0", "(\'out_prncp\', 5)_0", "(\'revol_bal\', 1)_0", "(\'revol_util\', 1)_0", "(\'inflation\', 4)_0", "(\'num_bc_tl_static\', 0)_0", "(\'num_tl_op_past_12m_static\', 0)_0", "(\'verification_status_joint_static\', 0)_0", "(\'total_pymnt\', 5)_0", "(\'acc_now_delinq_static\', 0)_0", "(\'tot_hi_cred_lim_static\', 0)_0", "(\'revol_bal\', 3)_0", "(\'mths_since_recent_revol_delinq_static\', 0)_0", "(\'policy_code_static\', 0)_

In [ ]:
# -------------------------
# 6. Inference
# -------------------------
# Use DBNInference to compute the probability distribution for default_indicator at a future time slice,
# for example, at time slice 6.
dbn_infer = DBNInference(model)
query_result = dbn_infer.query(variables=[('default_indicator', 6)])
print("\nProbability distribution for default_indicator at time slice 6:")
print(query_result)

In [53]:
import pandas as pd
from pgmpy.models import DynamicBayesianNetwork as DBN
from pgmpy.estimators import MaximumLikelihoodEstimator

# -------------------------
# 1. Load Data
# -------------------------
# Assume the CSV contains only the following columns:
static_cols = [
    'loan_amnt','grade','sub_grade','emp_length','home_ownership','annual_inc',
    'delinq_2yrs','inq_last_6mths','mths_since_last_delinq','mths_since_last_record',
    'open_acc','total_acc','acc_now_delinq','tot_coll_amt','tot_cur_bal',
    'open_acc_6m','open_act_il','open_il_12m','open_il_24m',
    'issue_quarter','approx_age','age_group'
]

# Read the CSV (adjust the file name as needed)
data = df_raw.copy()

# -------------------------
# 2. Prepare Data for DBN
# -------------------------
# Since these features are static, we assign them to time slice 0.
# We create a MultiIndex on the columns with time 0.
data.columns = pd.MultiIndex.from_product([data.columns.tolist(), [0]])

# -------------------------
# 3. Build a Minimal DBN Structure
# -------------------------
# Create a DBN model and add nodes for each static variable at time slice 0.
model = DBN()
for var in data.columns.get_level_values(0).unique():
    model.add_node((var, 0))

# For a minimal model, we can add a few example edges.
# For instance, let’s assume that 'annual_inc' influences 'loan_amnt' and 'approx_age'.
model.add_edge(('annual_inc', 0), ('loan_amnt', 0))
model.add_edge(('annual_inc', 0), ('approx_age', 0))

# -------------------------
# 4. Train the Model Using MLE
# -------------------------
# With only one time slice, this fits the CPDs for a static Bayesian network.
model.fit(data, estimator="MLE")

# -------------------------
# 5. Inspect the Learned CPDs
# -------------------------
print(model.cpds)


[]


In [59]:
import pandas as pd
from pgmpy.models import DynamicBayesianNetwork as DBN
from pgmpy.estimators import MaximumLikelihoodEstimator

# -------------------------
# 1. Load Data
# -------------------------
# Assume your CSV "credit_data_dynamic.csv" contains columns like:
# 'total_pymnt_t1','total_pymnt_t2', ... 'default_indicator_t1','default_indicator_t2', etc.
dynamic_cols = [
    'total_pymnt_t1','total_pymnt_t2','total_pymnt_t3','total_pymnt_t4','total_pymnt_t5','total_pymnt_t6',
    'revol_bal_t1','revol_bal_t2','revol_bal_t3','revol_bal_t4','revol_bal_t5','revol_bal_t6',
    'last_pymnt_amnt_t1','last_pymnt_amnt_t2','last_pymnt_amnt_t3','last_pymnt_amnt_t4','last_pymnt_amnt_t5','last_pymnt_amnt_t6',
    'default_indicator_t1','default_indicator_t2','default_indicator_t3','default_indicator_t4','default_indicator_t5','default_indicator_t6',
    'fico_score_t1','fico_score_t2','fico_score_t3','fico_score_t4','fico_score_t5','fico_score_t6',
    'revol_util_t1','revol_util_t2','revol_util_t3','revol_util_t4','revol_util_t5','revol_util_t6'
]

# Read only the dynamic columns from the CSV
data = pd.read_csv('bnpl_loans_with_age_change.csv', usecols=dynamic_cols)

# -------------------------
# 2. Reshape Data into Time Slices
# -------------------------
# We want to remap _t1 -> time slice 0, _t2 -> time slice 1, ..., _t6 -> time slice 5.
# For each time slice, we remove the "_t{t}" suffix and create a DataFrame with MultiIndex columns.
slices = {}
for t in range(1, 7):
    new_t = t - 1  # remap: t1 becomes 0, t2 becomes 1, etc.
    # Get all columns for this time slice
    cols_t = [col for col in dynamic_cols if col.endswith(f"_t{t}")]
    if not cols_t:
        continue
    df_t = data[cols_t].copy()
    # Remove the '_t{t}' suffix so that the variable names become consistent across time slices
    df_t = df_t.rename(columns=lambda x: x.rsplit('_t', 1)[0])
    # Assign a MultiIndex: each column becomes (variable, new_t)
    df_t.columns = pd.MultiIndex.from_tuples([(col, new_t) for col in df_t.columns])
    slices[new_t] = df_t

# Concatenate all slices along the column axis
combined_df = pd.concat([slices[t] for t in sorted(slices.keys())], axis=1)

# Debug: print the column names (should be tuples like ('total_pymnt', 0), ('total_pymnt', 1), etc.)
print("Combined DataFrame columns:")
print(combined_df.columns.tolist())

# -------------------------
# 3. Build the DBN Structure (Dynamic Nodes Only)
# -------------------------
model = DBN()

# Get the list of dynamic variable names (assumed to be the same across time slices)
dynamic_vars = slices[0].columns.get_level_values(0).unique()

# Add nodes for each dynamic variable at each time slice (0 through 5)
for t in range(0, 6):
    for var in dynamic_vars:
        model.add_node((var, t))

# Example: define some intra-slice dependencies.
# For instance, we might assume that within each time slice, 'fico_score' influences 'default_indicator',
# and 'last_pymnt_amnt' and 'revol_util' also influence 'default_indicator'.
for t in range(0, 6):
    if 'fico_score' in dynamic_vars and 'default_indicator' in dynamic_vars:
        model.add_edge(('fico_score', t), ('default_indicator', t))
    if 'last_pymnt_amnt' in dynamic_vars and 'default_indicator' in dynamic_vars:
        model.add_edge(('last_pymnt_amnt', t), ('default_indicator', t))
    if 'revol_util' in dynamic_vars and 'default_indicator' in dynamic_vars:
        model.add_edge(('revol_util', t), ('default_indicator', t))

# Define inter-slice dependencies: for each variable, add an edge from time t to time t+1.
for t in range(0, 5):
    for var in dynamic_vars:
        model.add_edge((var, t), (var, t+1))
    # Optionally, propagate the default indicator over time:
    if 'default_indicator' in dynamic_vars:
        model.add_edge(('default_indicator', t), ('default_indicator', t+1))

print("\nDBN structure (edges):")
print(model.edges())

# -------------------------
# 4. Train the DBN Using MLE
# -------------------------
# Fit the model on the combined DataFrame (which has columns labeled by (variable, time))
model.fit(combined_df, estimator="MLE")

# -------------------------
# 5. Inspect the Learned CPDs
# -------------------------
print("\nLearned CPDs:")
for cpd in model.cpds:
    print(cpd)

Combined DataFrame columns:
[('total_pymnt', 0), ('revol_bal', 0), ('last_pymnt_amnt', 0), ('default_indicator', 0), ('fico_score', 0), ('revol_util', 0), ('total_pymnt', 1), ('revol_bal', 1), ('last_pymnt_amnt', 1), ('default_indicator', 1), ('fico_score', 1), ('revol_util', 1), ('total_pymnt', 2), ('revol_bal', 2), ('last_pymnt_amnt', 2), ('default_indicator', 2), ('fico_score', 2), ('revol_util', 2), ('total_pymnt', 3), ('revol_bal', 3), ('last_pymnt_amnt', 3), ('default_indicator', 3), ('fico_score', 3), ('revol_util', 3), ('total_pymnt', 4), ('revol_bal', 4), ('last_pymnt_amnt', 4), ('default_indicator', 4), ('fico_score', 4), ('revol_util', 4), ('total_pymnt', 5), ('revol_bal', 5), ('last_pymnt_amnt', 5), ('default_indicator', 5), ('fico_score', 5), ('revol_util', 5)]

DBN structure (edges):
[(<DynamicNode(fico_score, 0) at 0x329d40700>, <DynamicNode(default_indicator, 0) at 0x315ca8df0>), (<DynamicNode(fico_score, 0) at 0x329d40700>, <DynamicNode(fico_score, 1) at 0x3318d1d50>),

KeyError: '["(\'total_pymnt\', 1)_0", "(\'revol_bal\', 2)_0", "(\'revol_bal\', 5)_0", "(\'revol_util\', 2)_0", "(\'fico_score\', 1)_0", "(\'last_pymnt_amnt\', 2)_0", "(\'revol_util\', 5)_0", "(\'fico_score\', 4)_0", "(\'last_pymnt_amnt\', 5)_0", "(\'default_indicator\', 1)_0", "(\'default_indicator\', 4)_0", "(\'total_pymnt\', 0)_0", "(\'total_pymnt\', 3)_0", "(\'revol_bal\', 4)_0", "(\'revol_bal\', 1)_0", "(\'revol_util\', 1)_0", "(\'last_pymnt_amnt\', 1)_0", "(\'fico_score\', 0)_0", "(\'revol_util\', 4)_0", "(\'fico_score\', 3)_0", "(\'last_pymnt_amnt\', 4)_0", "(\'default_indicator\', 3)_0", "(\'default_indicator\', 0)_0", "(\'total_pymnt\', 2)_0", "(\'total_pymnt\', 5)_0", "(\'revol_bal\', 0)_0", "(\'revol_bal\', 3)_0", "(\'revol_util\', 0)_0", "(\'fico_score\', 2)_0", "(\'last_pymnt_amnt\', 3)_0", "(\'last_pymnt_amnt\', 0)_0", "(\'revol_util\', 3)_0", "(\'fico_score\', 5)_0", "(\'default_indicator\', 5)_0", "(\'default_indicator\', 2)_0", "(\'total_pymnt\', 4)_0", "(\'total_pymnt\